# 03 — XGBoost Hyperparameter Sweep  —  GOOGLE COLAB (single GPU)
## Hackathon IBM — Détection de Fraude Bancaire

**Notebook prêt-à-run sur Colab T4/L4/A100**.

### Ce que fait ce notebook
1. Vérifie le GPU (`nvidia-smi` + bench CPU vs GPU XGBoost).
2. **Prépare** le fichier `train_original_ratio.parquet` via `preprocess_pipeline.preprocess_dataset(...)` → `prepared_train_original_ratio.parquet`.
3. Lance une **grille de 12 configurations XGBoost** (shallow, deep, régularisé, agressif, etc.) avec **barre de progression en %** et **monitoring VRAM**.
4. Logue chaque run dans `logs_training_colab/experiment_results.csv`, sauve les modèles `.ubj`, affiche leaderboard + graphiques.

### Fichiers à uploader sur Colab
- `train_original_ratio.parquet`
- `prepared_test_050.0_pct.parquet`
- `preprocess_pipeline.py`


---
## 0. Activer le GPU sur Colab

Avant tout : `Runtime` → `Change runtime type` → `Hardware accelerator: GPU`.


In [ ]:
!nvidia-smi


---
## 1. Installation


In [ ]:
!pip install -q xgboost pyarrow tqdm


---
## 2. Imports & vérification CUDA


In [ ]:
import os, re, gc, time, json, threading, warnings, subprocess, sys
from pathlib import Path
from typing import Tuple, Dict, List

import numpy as np
import pandas as pd

from sklearn.metrics import (f1_score, recall_score, precision_score, accuracy_score,
                              roc_auc_score, average_precision_score, confusion_matrix)

import xgboost as xgb
import torch
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)

CUDA_AVAILABLE = torch.cuda.is_available()
N_GPUS = torch.cuda.device_count() if CUDA_AVAILABLE else 0
XGB_VERSION = tuple(int(x) for x in xgb.__version__.split('.')[:2])

print('=== Environment ===')
print(f'pandas  : {pd.__version__}')
print(f'torch   : {torch.__version__}')
print(f'xgboost : {xgb.__version__}')
print(f'CUDA    : {CUDA_AVAILABLE} (devices={N_GPUS})')
if CUDA_AVAILABLE:
    for i in range(N_GPUS):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU[{i}] : {p.name}  |  VRAM totale : {p.total_memory/1024**3:.1f} GB')
else:
    raise RuntimeError('CUDA NON DISPONIBLE. Active le GPU via Runtime -> Change runtime type.')


---
## 3. Preuve que XGBoost utilise bien le GPU (mini benchmark)


In [ ]:
from sklearn.datasets import make_classification

print('Generation d un dataset synthetique (150k x 40)...')
X_b, y_b = make_classification(n_samples=150_000, n_features=40, n_informative=25,
                                n_redundant=5, weights=[0.98, 0.02], random_state=42)
X_b = pd.DataFrame(X_b.astype(np.float32))

m_cpu = xgb.XGBClassifier(n_estimators=400, tree_method='hist', device='cpu',
                           max_depth=6, verbosity=0)
t0 = time.perf_counter(); m_cpu.fit(X_b, y_b); t_cpu = time.perf_counter() - t0
print(f'  XGBoost CPU : {t_cpu:6.2f}s')

torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
if XGB_VERSION >= (2, 0):
    m_gpu = xgb.XGBClassifier(n_estimators=400, tree_method='hist', device='cuda:0',
                               max_depth=6, verbosity=0)
else:
    m_gpu = xgb.XGBClassifier(n_estimators=400, tree_method='gpu_hist', gpu_id=0,
                               max_depth=6, verbosity=0)
t0 = time.perf_counter(); m_gpu.fit(X_b, y_b); t_gpu = time.perf_counter() - t0
vram_peak = torch.cuda.max_memory_allocated() / 1024**2
print(f'  XGBoost GPU : {t_gpu:6.2f}s  (VRAM peak ~ {vram_peak:.0f} MB)')
print(f'\n  Speedup GPU / CPU : x{t_cpu/t_gpu:.1f}')

if t_gpu < t_cpu * 0.8 and vram_peak > 20:
    print('\nOK : GPU bien utilise par XGBoost.')
else:
    print('\nVerif non concluante ici. Le monitoring VRAM pendant les vrais runs tranchera.')

del X_b, y_b, m_cpu, m_gpu; gc.collect(); torch.cuda.empty_cache()


---
## 4. Upload des fichiers

Trois options — choisis-en une :


In [ ]:
# --- OPTION A : Upload direct (decommente les 2 lignes) ---
# Upload : train_original_ratio.parquet, prepared_test_050.0_pct.parquet, preprocess_pipeline.py
# from google.colab import files
# files.upload()

# --- OPTION B : Google Drive (decommente et ajuste le chemin) ---
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/hackaton_ibm/train_original_ratio.parquet /content/drive/MyDrive/hackaton_ibm/prepared_test_050.0_pct.parquet /content/drive/MyDrive/hackaton_ibm/preprocess_pipeline.py /content/

# --- OPTION C : Fichiers deja dans /content ---
DATA_DIR = Path('.')

required = ['train_original_ratio.parquet', 'prepared_test_050.0_pct.parquet', 'preprocess_pipeline.py']
for f in required:
    p = DATA_DIR / f
    if p.exists():
        print(f'  OK      : {f:<42}  ({p.stat().st_size/1024**2:7.2f} MB)')
    else:
        print(f'  MANQUE  : {f}')

missing = [f for f in required if not (DATA_DIR / f).exists()]
assert not missing, f'Upload d abord : {missing}'


---
## 5. Configuration globale


In [ ]:
RAW_TRAIN_FILE      = 'train_original_ratio.parquet'
PREPARED_TRAIN_FILE = 'prepared_train_original_ratio.parquet'
TEST_FILE           = 'prepared_test_050.0_pct.parquet'

LOG_DIR     = Path('logs_training_colab')
MODELS_DIR  = LOG_DIR / 'models'
RESULTS_CSV = LOG_DIR / 'experiment_results.csv'
LOG_DIR.mkdir(exist_ok=True, parents=True)
MODELS_DIR.mkdir(exist_ok=True, parents=True)

TARGET       = 'is_fraud'
RANDOM_STATE = 42
XGB_VERBOSE_PERIOD = 100   # print toutes les 100 iters (preuve d activite)

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if CUDA_AVAILABLE:
    torch.cuda.manual_seed_all(RANDOM_STATE)

print(f'Logs    -> {LOG_DIR.resolve()}')
print(f'Models  -> {MODELS_DIR.resolve()}')


---
## 6. Préparation de `train_original_ratio.parquet`

Applique `preprocess_dataset` et sauve en `prepared_train_original_ratio.parquet`.
Cellule **idempotente** : skip si le fichier préparé existe déjà.


In [ ]:
sys.path.insert(0, str(DATA_DIR.resolve()))
from preprocess_pipeline import preprocess_dataset  # noqa: E402

prepared_path = DATA_DIR / PREPARED_TRAIN_FILE

if prepared_path.exists():
    print(f'OK : {PREPARED_TRAIN_FILE} existe deja ({prepared_path.stat().st_size/1024**2:.2f} MB) - skip preprocessing.')
else:
    raw_path = DATA_DIR / RAW_TRAIN_FILE
    print(f'Chargement de {RAW_TRAIN_FILE} ...')
    t0 = time.perf_counter()
    df_raw = pd.read_parquet(raw_path)
    print(f'  shape    : {df_raw.shape}  |  memoire : {df_raw.memory_usage(deep=True).sum()/1024**2:.1f} MB')

    print('\nApplication de preprocess_dataset(...) ...')
    df_prep = preprocess_dataset(df_raw, verbose=True)
    print(f'\n  shape finale : {df_prep.shape}')
    print(f'  memoire      : {df_prep.memory_usage(deep=True).sum()/1024**2:.1f} MB')
    print(f'  is_fraud     : {df_prep[TARGET].mean()*100:.4f} %  ({int(df_prep[TARGET].sum())} positifs / {len(df_prep)})')

    print(f'\nSauvegarde -> {prepared_path} ...')
    df_prep.to_parquet(prepared_path, index=False, compression='snappy')
    print(f'Termine en {time.perf_counter()-t0:.1f} s')

    del df_raw, df_prep; gc.collect()


---
## 7. Chargement train + test


In [ ]:
def load_dataset(path) -> Tuple[pd.DataFrame, pd.Series, List[str]]:
    df = pd.read_parquet(path)
    y = df[TARGET].astype(np.int8)
    X = df.drop(columns=[TARGET])
    cat_features = [c for c, dt in X.dtypes.items() if str(dt) == 'category']
    return X, y, cat_features


def align_categories(X_train, X_test, cat_features):
    X_train, X_test = X_train.copy(), X_test.copy()
    for c in cat_features:
        all_cats = pd.api.types.union_categoricals(
            [X_train[c], X_test[c]], sort_categories=True).categories
        X_train[c] = pd.Categorical(X_train[c], categories=all_cats)
        X_test[c]  = pd.Categorical(X_test[c],  categories=all_cats)
    return X_train, X_test


print('Chargement du train...')
X_train, y_train, cat_features = load_dataset(DATA_DIR / PREPARED_TRAIN_FILE)
print(f'  shape   : {X_train.shape}')
print(f'  fraude  : {y_train.mean()*100:.4f} %  ({int(y_train.sum())} positifs / {len(y_train)})')
print(f'  cat cols: {len(cat_features)}\n')

print('Chargement du test...')
X_test, y_test, _ = load_dataset(DATA_DIR / TEST_FILE)
print(f'  shape   : {X_test.shape}')
print(f'  fraude  : {y_test.mean()*100:.4f} %  ({int(y_test.sum())} positifs / {len(y_test)})\n')

print('Alignement des categories train/test...')
X_train, X_test = align_categories(X_train, X_test, cat_features)

BASE_SPW = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
print(f'\n  scale_pos_weight de base (neg/pos) : {BASE_SPW:.1f}')


---
## 8. Grille de 12 configurations XGBoost


In [ ]:
def gpu_params():
    if XGB_VERSION >= (2, 0):
        return dict(tree_method='hist', device='cuda:0')
    else:
        return dict(tree_method='gpu_hist', gpu_id=0, predictor='gpu_predictor')


BASE = dict(
    objective='binary:logistic',
    eval_metric='aucpr',
    random_state=RANDOM_STATE,
    enable_categorical=True,
    n_jobs=-1,
    early_stopping_rounds=100,
    verbosity=0,
    **gpu_params(),
)

CONFIGS = [
    {'name': '01_baseline',          'params': dict(n_estimators=1500, max_depth=6,  learning_rate=0.05,  subsample=0.85, colsample_bytree=0.85, min_child_weight=1.0,  reg_lambda=1.0, reg_alpha=0.0, gamma=0.0, scale_pos_weight=BASE_SPW)},
    {'name': '02_shallow_fast',      'params': dict(n_estimators=2000, max_depth=4,  learning_rate=0.10,  subsample=0.90, colsample_bytree=0.90, min_child_weight=1.0,  reg_lambda=1.0, reg_alpha=0.0, gamma=0.0, scale_pos_weight=BASE_SPW)},
    {'name': '03_deep_slow',         'params': dict(n_estimators=3000, max_depth=10, learning_rate=0.03,  subsample=0.80, colsample_bytree=0.80, min_child_weight=2.0,  reg_lambda=1.5, reg_alpha=0.0, gamma=0.0, scale_pos_weight=BASE_SPW)},
    {'name': '04_deep_aggressive',   'params': dict(n_estimators=2000, max_depth=12, learning_rate=0.08,  subsample=0.85, colsample_bytree=0.85, min_child_weight=1.0,  reg_lambda=1.0, reg_alpha=0.0, gamma=0.0, scale_pos_weight=BASE_SPW)},
    {'name': '05_high_regularized',  'params': dict(n_estimators=2000, max_depth=8,  learning_rate=0.05,  subsample=0.75, colsample_bytree=0.75, min_child_weight=5.0,  reg_lambda=5.0, reg_alpha=2.0, gamma=1.0, scale_pos_weight=BASE_SPW)},
    {'name': '06_no_regularization', 'params': dict(n_estimators=1500, max_depth=8,  learning_rate=0.05,  subsample=1.00, colsample_bytree=1.00, min_child_weight=0.5,  reg_lambda=0.0, reg_alpha=0.0, gamma=0.0, scale_pos_weight=BASE_SPW)},
    {'name': '07_low_subsample',     'params': dict(n_estimators=2000, max_depth=7,  learning_rate=0.05,  subsample=0.60, colsample_bytree=0.60, min_child_weight=1.0,  reg_lambda=1.0, reg_alpha=0.0, gamma=0.0, scale_pos_weight=BASE_SPW)},
    {'name': '08_spw_x2',            'params': dict(n_estimators=1500, max_depth=6,  learning_rate=0.05,  subsample=0.85, colsample_bytree=0.85, min_child_weight=1.0,  reg_lambda=1.0, reg_alpha=0.0, gamma=0.0, scale_pos_weight=BASE_SPW*2)},
    {'name': '09_spw_neutral',       'params': dict(n_estimators=1500, max_depth=6,  learning_rate=0.05,  subsample=0.85, colsample_bytree=0.85, min_child_weight=1.0,  reg_lambda=1.0, reg_alpha=0.0, gamma=0.0, scale_pos_weight=1.0)},
    {'name': '10_spw_half',          'params': dict(n_estimators=1500, max_depth=6,  learning_rate=0.05,  subsample=0.85, colsample_bytree=0.85, min_child_weight=1.0,  reg_lambda=1.0, reg_alpha=0.0, gamma=0.0, scale_pos_weight=BASE_SPW/2)},
    {'name': '11_min_child_heavy',   'params': dict(n_estimators=2000, max_depth=8,  learning_rate=0.05,  subsample=0.85, colsample_bytree=0.85, min_child_weight=10.0, reg_lambda=1.0, reg_alpha=0.0, gamma=0.5, scale_pos_weight=BASE_SPW)},
    {'name': '12_long_balanced',     'params': dict(n_estimators=4000, max_depth=7,  learning_rate=0.025, subsample=0.85, colsample_bytree=0.85, min_child_weight=2.0,  reg_lambda=2.0, reg_alpha=0.5, gamma=0.2, scale_pos_weight=BASE_SPW)},
]

print(f'=== {len(CONFIGS)} configurations definies ===')
for c in CONFIGS:
    p = c['params']
    print(f"  {c['name']:<22}  d={p['max_depth']:>2}  lr={p['learning_rate']:.3f}  "
          f"n={p['n_estimators']:>4}  sub={p['subsample']}  cs={p['colsample_bytree']}  "
          f"spw={p['scale_pos_weight']:.1f}  L2={p['reg_lambda']}  L1={p['reg_alpha']}")


---
## 9. Monitor VRAM en arrière-plan


In [ ]:
class VRAMMonitor:
    def __init__(self, interval: float = 2.0):
        self.interval = interval
        self._stop = threading.Event()
        self._thread = None
        self.samples = []

    def _loop(self):
        t0 = time.perf_counter()
        while not self._stop.is_set():
            try:
                alloc = torch.cuda.memory_allocated() / 1024**2
                reser = torch.cuda.memory_reserved()  / 1024**2
                self.samples.append((time.perf_counter() - t0, alloc, reser))
            except Exception:
                pass
            self._stop.wait(self.interval)

    def start(self):
        self.samples = []; self._stop.clear()
        if CUDA_AVAILABLE:
            torch.cuda.reset_peak_memory_stats()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()

    def stop(self):
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=self.interval + 1)

    def peak_mb(self):
        if not self.samples:
            return 0.0
        return max(s[1] for s in self.samples)


---
## 10. Helpers metrics & logging


In [ ]:
def compute_metrics(y_true, y_pred, y_proba):
    metrics = {
        'F1':        f1_score(y_true, y_pred, zero_division=0),
        'Recall':    recall_score(y_true, y_pred, zero_division=0),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Accuracy':  accuracy_score(y_true, y_pred),
        'PR-AUC':    average_precision_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
        'ROC-AUC':   roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
    }
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
    metrics.update({'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)})
    return metrics


def append_result_row(row, csv_path):
    pd.DataFrame([row]).to_csv(csv_path, mode='a', header=not csv_path.exists(), index=False)


---
## 11. Boucle principale — barre de progression en % + VRAM


In [ ]:
def train_one_config(name, params, X_train, y_train, X_test, y_test, pbar):
    full_params = {**BASE, **params}
    model = xgb.XGBClassifier(**full_params)

    mon = VRAMMonitor(interval=2.0); mon.start()
    t0 = time.perf_counter()
    try:
        model.fit(X_train, y_train,
                  eval_set=[(X_test, y_test)],
                  verbose=XGB_VERBOSE_PERIOD)
    finally:
        mon.stop()
    train_time = time.perf_counter() - t0

    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred  = (y_proba >= 0.5).astype(int)
    metrics = compute_metrics(y_test.values, y_pred, y_proba)

    vram_peak = mon.peak_mb()
    model.save_model(str(MODELS_DIR / f'xgb_{name}.ubj'))

    row = {
        'Config':               name,
        'n_estimators':         params.get('n_estimators'),
        'max_depth':            params.get('max_depth'),
        'learning_rate':        params.get('learning_rate'),
        'subsample':            params.get('subsample'),
        'colsample_bytree':     params.get('colsample_bytree'),
        'min_child_weight':     params.get('min_child_weight'),
        'reg_lambda':           params.get('reg_lambda'),
        'reg_alpha':            params.get('reg_alpha'),
        'gamma':                params.get('gamma'),
        'scale_pos_weight':     round(params.get('scale_pos_weight', 1.0), 3),
        'best_iteration':       getattr(model, 'best_iteration', None),
        **{k: round(v, 6) if isinstance(v, float) else v for k, v in metrics.items()},
        'Training_Time_Sec':    round(train_time, 2),
        'VRAM_Peak_MB':         round(vram_peak, 1),
    }
    append_result_row(row, RESULTS_CSV)

    pbar.write(
        f'  OK  {name:<22} | F1={metrics["F1"]:.4f} | PR-AUC={metrics["PR-AUC"]:.4f} | '
        f'Recall={metrics["Recall"]:.4f} | Prec={metrics["Precision"]:.4f} | '
        f'best_it={row["best_iteration"]} | VRAM={vram_peak:>5.0f}MB | {train_time:5.1f}s'
    )

    del model; gc.collect(); torch.cuda.empty_cache()
    return row


In [ ]:
if RESULTS_CSV.exists():
    RESULTS_CSV.unlink()

print(f'=== Lancement de {len(CONFIGS)} runs XGBoost sur {PREPARED_TRAIN_FILE} ===\n')
t_all = time.perf_counter()
best_f1 = 0.0; best_cfg = None

with tqdm(total=len(CONFIGS), desc='Sweep XGBoost', unit='cfg',
          bar_format='{l_bar}{bar:30}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}] {postfix}'
         ) as pbar:
    for cfg in CONFIGS:
        pbar.set_description(f'Sweep XGBoost  |  {cfg["name"]}')
        try:
            row = train_one_config(cfg['name'], cfg['params'],
                                    X_train, y_train, X_test, y_test, pbar)
            if row['F1'] > best_f1:
                best_f1 = row['F1']; best_cfg = cfg['name']
            pbar.set_postfix({'best_f1': f'{best_f1:.4f} ({best_cfg})'})
        except Exception as e:
            pbar.write(f'  ERR {cfg["name"]:<22} | {type(e).__name__}: {e}')
            append_result_row({'Config': cfg['name'],
                                'Error': f'{type(e).__name__}: {e}'}, RESULTS_CSV)
        pbar.update(1)

print(f'\n=== TERMINE en {(time.perf_counter()-t_all)/60:.1f} min ===')
print(f'Resultats -> {RESULTS_CSV.resolve()}')
print(f'Modeles   -> {MODELS_DIR.resolve()}')


---
## 12. Preuve GPU finale


In [ ]:
!nvidia-smi
print('\nDans experiment_results.csv, la colonne VRAM_Peak_MB doit etre > 100-500 MB pour chaque config.')
print('Si elle est = 0, le GPU n a pas ete utilise.')


---
## 13. Leaderboard


In [ ]:
results = pd.read_csv(RESULTS_CSV)
print(f'Resultats : {results.shape}\n')

if 'PR-AUC' in results.columns:
    lb = (results.dropna(subset=['PR-AUC'])
                 .sort_values('PR-AUC', ascending=False)
                 .loc[:, ['Config', 'F1', 'Recall', 'Precision', 'PR-AUC', 'ROC-AUC',
                          'TP', 'FP', 'FN', 'TN', 'best_iteration',
                          'Training_Time_Sec', 'VRAM_Peak_MB']]
                 .reset_index(drop=True))
    print('=== Classement par PR-AUC ===')
    print(lb.to_string(index=False))
    print('\n=== Classement par F1 ===')
    print(lb.sort_values('F1', ascending=False).to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt

if 'PR-AUC' in results.columns and not results['PR-AUC'].isna().all():
    sub = results.dropna(subset=['PR-AUC']).sort_values('PR-AUC', ascending=True)
    metrics_show = ['F1', 'Recall', 'Precision', 'PR-AUC']
    fig, axes = plt.subplots(1, len(metrics_show), figsize=(4*len(metrics_show), max(5, 0.35*len(sub))))
    for ax, m in zip(axes, metrics_show):
        ax.barh(sub['Config'], sub[m])
        ax.set_xlabel(m); ax.set_title(m); ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(LOG_DIR / 'leaderboard.png', dpi=110, bbox_inches='tight')
    plt.show()


---
## 14. Télécharger les résultats


In [ ]:
!zip -r logs_training_colab.zip logs_training_colab > /dev/null
print('Archive creee : logs_training_colab.zip')

# Decommente pour telecharger :
# from google.colab import files
# files.download('logs_training_colab.zip')
